In [18]:
import os
import json
from dataclasses import dataclass
from pydantic import BaseModel
from typing import Literal

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """One provider described as pure DATA (same design as Notebook 1)."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file.")


def build_client(provider: Provider):
    from openai import OpenAI

    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_key() -> bool:
    return any(os.environ.get(p.env_var) for p in PROVIDERS)



def llm_reply(prompt: str, *, max_tokens: int = 400) -> str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return result.choices[0].message.content

In [3]:
SYSTEM_PROMPT = """
You are a product-review sentiment analyst.

Read each review the user sends and classify its sentiment. We should also attach a confidence score for the sentiment
between 0 to 1 and a short explanation for the sentiment as a reason.

For each review, respond with ONLY a JSON object -  no other text or comments - in exactly the following shape:

{
    "sentiment": "positive" | "negative" | "neutral",
    "confidence": 0.0 to 1.0,
    "reason": "short explanation for the sentiment"
}

""".strip()

In [4]:
def analyze_review(incoming_review: str) -> str:
    """Send review to LLM and ask it to classify the sentiment."""
    provider = select_provider()
    client = build_client(provider)
    
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=100,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": incoming_review},
        ],
    )

    return result.choices[0].message.content.strip()

In [5]:
REVIEWS = [
        "Absolutely love these headphones — great sound and the battery lasts all day!",
        "Arrived broken and customer service never replied. Very disappointed.",
        "It works. Nothing special, nothing terrible.",
    ]

In [ ]:
review1 =analyze_review(REVIEWS[0])

'{\n    "sentiment": "positive",\n    "confidence": 0.95,\n    "reason": "The user expresses strong affection for the headphones, highlighting great sound quality and long battery life."\n}'

In [ ]:
review2 = analyze_review(REVIEWS[1])

'{\n    "sentiment": "negative",\n    "confidence": 0.9,\n    "reason": "The review expresses dissatisfaction due to a broken item and lack of customer service response."\n}'

In [9]:
review3 = analyze_review(REVIEWS[2])

In [19]:
class SentimentResult(BaseModel):
    """Sentiment result from the LLM."""

    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    reason: str


In [20]:
def validate_sentiment_reply(raw_reply: str) -> SentimentResult | str:
    try:
        parsed = json.loads(raw_reply)
        return SentimentResult(**parsed)
    except json.JSONDecodeError:
        return f"Invalid JSON: {raw_reply}"



In [21]:
result = validate_sentiment_reply(review3)

In [23]:
print(type(result))

<class '__main__.SentimentResult'>


In [25]:
def route_by_sentiment(result: SentimentResult) -> str:
    """Route the review based on the sentiment."""
    if result.sentiment == "positive":
        return "Positive review"
    elif result.sentiment == "negative":
        return "Negative review, connect with customer service"
    else:
        return "Neutral review, no action needed"


route_by_sentiment(result)



'Neutral review, no action needed'